# Actividad 2 — Principio de Sustitución de Liskov (LSP)

**Estudiante:** Andrés Felipe Luna Camargo  
**Dominio:** Control Operativo de Unidades de Enfriamiento en Planta

---



## 1. Ejemplo Incorrecto (Violando LSP)

Imaginemos que tenemos una clase padre `UnidadCompresorMecanico` que modela cuartos fríos tradicionales con motor a gas refrigerante.  
Luego creamos una subclase `CamaraCriogenica` que enfría con nitrógeno líquido ($LN_2$). Como no tiene compresor mecánico:
1. Si alguien llama a `encender_compresor()`, lanza una excepción `RuntimeError`.
2. Si alguien llama a `obtener_presion_psi()`, devuelve `None` en lugar de un número `float`.



In [ ]:
# Ejemplo de herencia incorrecta que rompe el principio de Liskov

class UnidadCompresorMecanico:
    def __init__(self, id_equipo: str, temp_objetivo: float, presion_psi: float) -> None:
        self.id_equipo: str = id_equipo
        self.temp_objetivo: float = temp_objetivo
        self.presion_psi: float = presion_psi
        self.encendido: bool = False

    def encender_compresor(self) -> str:
        self.encendido = True
        return f"[{self.id_equipo}] Compresor mecánico encendido a {self.presion_psi} PSI."

    def obtener_presion_psi(self) -> float:
        return self.presion_psi


class CamaraCriogenica(UnidadCompresorMecanico):
    def __init__(self, id_equipo: str, temp_objetivo: float, nivel_n2_litros: float) -> None:
        # Pone presion 0 porque no usa gas comprimido
        super().__init__(id_equipo, temp_objetivo, presion_psi=0.0)
        self.nivel_n2_litros: float = nivel_n2_litros

    def encender_compresor(self) -> str:
        # VIOLA LSP: Lanza error inesperado porque no tiene motor compresor
        raise RuntimeError(f"¡Error en {self.id_equipo}! Las unidades criogénicas no tienen compresor mecánico.")

    def obtener_presion_psi(self) -> float:
        # VIOLA LSP: Retorna None en vez de un float, dañando formatos matemáticos
        return None


Clases con herencia defectuosa creadas.


In [2]:
# Función cliente que espera trabajar con la clase padre
def panel_arranque_planta(lista_equipos: list) -> None:
    print("Iniciando ciclo de arranque en el parque frigorífico...")
    for equipo in lista_equipos:
        # El cliente asume que puede leer la presión y encender el compresor
        presion = equipo.obtener_presion_psi()
        print(f"Revisando equipo {equipo.id_equipo} con presión {presion:.1f} PSI")
        resultado = equipo.encender_compresor()
        print(resultado)


print("--- Probando Violación de LSP ---")
equipos = [
    UnidadCompresorMecanico("COMP-01", -18.0, 32.0),
    CamaraCriogenica("CRIO-99", -196.0, 500.0)  # Esta va a romper el programa
]

try:
    panel_arranque_planta(equipos)
except Exception as e:
    print(f"\n--> SE ROMPIÓ EL PROGRAMA: {type(e).__name__} -> {e}")
    print("Explicación: CamaraCriogenica no puede sustituir a UnidadCompresorMecanico.")


--- Probando Violación de LSP ---
Iniciando ciclo de arranque en el parque frigorífico...
Revisando equipo COMP-01 con presión 32.0 PSI
[COMP-01] Compresor mecánico encendido a 32.0 PSI.

--> SE ROMPIÓ EL PROGRAMA: TypeError -> unsupported format string passed to NoneType.__format__
Explicación: CamaraCriogenica no puede sustituir a UnidadCompresorMecanico.


## 2. Ejemplo Correcto (Aplicando LSP)

Para cumplir LSP, no debemos forzar a una cámara criogénica a heredar métodos de compresores mecánicos.  
Creamos una clase base abstracta `UnidadTermica(ABC)` que solo exige lo que **todas** las tecnologías de frío realmente pueden cumplir:
1. `iniciar_enfriamiento()`: cada una enfría a su manera (una prende el motor, la otra abre la válvula de nitrógeno).
2. `obtener_estado()`: devuelve un diccionario con datos claros y tipos consistentes sin devolver `None` ni lanzar excepciones.


In [ ]:
from abc import ABC, abstractmethod

#  Clase base con contrato universal para cualquier tecnología de frío
class UnidadTermica(ABC):
    def __init__(self, id_equipo: str, temp_objetivo: float) -> None:
        self.id_equipo: str = id_equipo
        self.temp_objetivo: float = temp_objetivo
        self.activo: bool = False

    @abstractmethod
    def iniciar_enfriamiento(self) -> str:
        pass

    @abstractmethod
    def obtener_estado(self) -> dict:
        pass


# Subclase A: Compresión Mecánica tradicional
class UnidadCompresion(UnidadTermica):
    def __init__(self, id_equipo: str, temp_objetivo: float, presion_psi: float, rpm_motor: int) -> None:
        super().__init__(id_equipo, temp_objetivo)
        self.presion_psi: float = presion_psi
        self.rpm_motor: int = rpm_motor

    def iniciar_enfriamiento(self) -> str:
        self.activo = True
        return f"[{self.id_equipo}] Motor arrancado a {self.rpm_motor} RPM (Presión: {self.presion_psi} PSI)"

    def ajustar_rpm(self, nuevas_rpm: int) -> None:
        self.rpm_motor = nuevas_rpm

    def obtener_estado(self) -> dict:
        return {
            "id": self.id_equipo,
            "tipo": "Compresor Mecánico",
            "setpoint": self.temp_objetivo,
            "activo": self.activo,
            "detalle_operacion": f"Presión gas: {self.presion_psi} PSI"
        }


# Subclase B: Criogenia con Nitrógeno Líquido
class UnidadCriogenica(UnidadTermica):
    def __init__(self, id_equipo: str, temp_objetivo: float, nivel_n2_litros: float, apertura_valvula_pct: float) -> None:
        super().__init__(id_equipo, temp_objetivo)
        self.nivel_n2_litros: float = nivel_n2_litros
        self.apertura_valvula_pct: float = apertura_valvula_pct

    def iniciar_enfriamiento(self) -> str:
        self.activo = True
        return f"[{self.id_equipo}] Válvula de N2 abierta al {self.apertura_valvula_pct}% (Tanque: {self.nivel_n2_litros} L)"

    def recargar_tanque(self, litros: float) -> None:
        self.nivel_n2_litros += litros

    def obtener_estado(self) -> dict:
        return {
            "id": self.id_equipo,
            "tipo": "Inyección Criogénica",
            "setpoint": self.temp_objetivo,
            "activo": self.activo,
            "detalle_operacion": f"Tanque N2: {self.nivel_n2_litros} Litros"
        }


Jerarquía correcta con LSP creada.


In [4]:
# Función cliente que ahora sí puede sustituir cualquier subclase sin fallar
def panel_control_general(lista_unidades: list) -> None:
    print("=== PANEL GENERAL DE CONTROL FRIGORÍFICO ===")
    for u in lista_unidades:
        # Sustitución perfecta: todas cumplen el contrato sin exceptions
        arranque = u.iniciar_enfriamiento()
        info = u.obtener_estado()
        print(arranque)
        print(f"   -> [{info['tipo']}] Setpoint: {info['setpoint']}°C | {info['detalle_operacion']}")
    print("-" * 65)


print("--- Probando Sustitución de Liskov (Cumple LSP) ---")
parque_equipos = [
    UnidadCompresion("COMP-CARNES-01", temp_objetivo=-18.0, presion_psi=28.5, rpm_motor=3200),
    UnidadCompresion("COMP-LACTEOS-02", temp_objetivo=4.0, presion_psi=42.0, rpm_motor=1750),
    UnidadCriogenica("CRIO-BIOBANCO-01", temp_objetivo=-196.0, nivel_n2_litros=1200.0, apertura_valvula_pct=30.0),
    UnidadCriogenica("CRIO-VACUNAS-02", temp_objetivo=-80.0, nivel_n2_litros=850.0, apertura_valvula_pct=15.0)
]

# Funciona de forma transparente para todas las unidades
panel_control_general(parque_equipos)


--- Probando Sustitución de Liskov (Cumple LSP) ---
=== PANEL GENERAL DE CONTROL FRIGORÍFICO ===
[COMP-CARNES-01] Motor arrancado a 3200 RPM (Presión: 28.5 PSI)
   -> [Compresor Mecánico] Setpoint: -18.0°C | Presión gas: 28.5 PSI
[COMP-LACTEOS-02] Motor arrancado a 1750 RPM (Presión: 42.0 PSI)
   -> [Compresor Mecánico] Setpoint: 4.0°C | Presión gas: 42.0 PSI
[CRIO-BIOBANCO-01] Válvula de N2 abierta al 30.0% (Tanque: 1200.0 L)
   -> [Inyección Criogénica] Setpoint: -196.0°C | Tanque N2: 1200.0 Litros
[CRIO-VACUNAS-02] Válvula de N2 abierta al 15.0% (Tanque: 850.0 L)
   -> [Inyección Criogénica] Setpoint: -80.0°C | Tanque N2: 850.0 Litros
-----------------------------------------------------------------
